# Valutazione di HunyuanPortrait, PersonaLive e Reface

**Prima di tutto:** Runtime → Cambia tipo di runtime → GPU (A100). Poi esegui le celle in ordine.

1. Setup e dati
2. Configurazione e modelli di face analysis (InsightFace)
3. NR-IQA sui ritagli di volto (immagini e frame)
4. Riepilogo IQA, grafici, test statistici
5. Metriche video (identità, jitter, flicker)
6. Riepilogo metriche video

In [ ]:
!nvidia-smi -L
!pip install -q pyiqa insightface onnxruntime-gpu opencv-python-headless pandas tqdm lpips scipy matplotlib
# Se dopo l'installazione compare un errore su numpy: Runtime -> Riavvia sessione, poi riesegui da qui.

from google.colab import drive
import os

drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Risultati finali.zip"
EXTRACT_DIR = "/content/dataset"

if not os.path.exists(EXTRACT_DIR):
    !unzip -qo "{ZIP_PATH}" -d "{EXTRACT_DIR}" -x "__MACOSX/*"
    print("Estrazione completata")

BASE_DIR = os.path.join(EXTRACT_DIR, "Risultati finali")
if not os.path.isdir(BASE_DIR):
    BASE_DIR = EXTRACT_DIR
print("BASE_DIR:", BASE_DIR)
print("Sottocartelle:", [d for d in sorted(os.listdir(BASE_DIR)) if not d.startswith('.') and d != '__MACOSX'])

In [ ]:
import os, cv2, numpy as np, pandas as pd, torch
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

SAVE_DIR = "/content/drive/MyDrive/risultati_valutazione"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_EXT = (".png", ".jpg", ".jpeg", ".webp")
VID_EXT = (".mp4", ".avi", ".mov", ".mkv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NESSUNA GPU")

def list_media(folder, exts):
    out = []
    for root, _, files in os.walk(folder):
        if "__MACOSX" in root:
            continue
        for f in files:
            if not f.startswith("._") and f.lower().endswith(exts):
                out.append(os.path.join(root, f))
    return sorted(out)

model_folders = [d for d in sorted(os.listdir(BASE_DIR))
                 if os.path.isdir(os.path.join(BASE_DIR, d)) and not d.startswith(".") and d != "__MACOSX"]
for mf in model_folders:
    p = os.path.join(BASE_DIR, mf)
    print(f"{mf}: {len(list_media(p, IMG_EXT))} immagini, {len(list_media(p, VID_EXT))} video")

# Face analysis: detection + embedding ArcFace + 68 landmark 3D (un'unica istanza condivisa)
from insightface.app import FaceAnalysis
import onnxruntime
print("ONNX providers disponibili:", onnxruntime.get_available_providers())

face_app = FaceAnalysis(name="buffalo_l",
                        allowed_modules=["detection", "recognition", "landmark_3d_68"],
                        providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
face_app.prepare(ctx_id=0, det_size=(640, 640))
try:
    print("Provider usato dal detector:", face_app.models["detection"].session.get_providers()[0])
except Exception:
    pass

def main_face(faces):
    return max(faces, key=lambda x: (x.bbox[2] - x.bbox[0]) * (x.bbox[3] - x.bbox[1]))

In [ ]:
import pyiqa

METRICS = ["topiq_nr-face", "musiq", "clipiqa", "brisque", "niqe", "maniqa"]
LOWER_IS_BETTER = {"brisque", "niqe"}
N_FRAMES = 32        # frame per video (distribuiti uniformemente)
CROP_SCALE = 1.6     # ampiezza del ritaglio rispetto al box del volto

print("Caricamento modelli IQA...")
models = {m: pyiqa.create_metric(m, device=device) for m in METRICS}
errors, first_err = Counter(), {}

def face_crop(bgr):
    h, w = bgr.shape[:2]
    faces = face_app.get(bgr)
    if not faces:
        return None
    x0, y0, x1, y1 = main_face(faces).bbox
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(x1 - x0, y1 - y0) * CROP_SCALE
    X0, Y0 = int(max(cx - side / 2, 0)), int(max(cy - side / 2, 0))
    X1, Y1 = int(min(cx + side / 2, w)), int(min(cy + side / 2, h))
    crop = bgr[Y0:Y1, X0:X1]
    return crop if min(crop.shape[:2]) >= 32 else None

@torch.no_grad()
def score_bgr(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).float().div(255).to(device)
    out = {}
    for name, m in models.items():
        try:
            out[name] = float(m(t).item())
        except Exception as e:
            out[name] = np.nan
            errors[name] += 1
            first_err.setdefault(name, repr(e)[:200])
    return out

def read_uniform(path, n):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    want = set(np.linspace(0, total - 1, min(n, total)).astype(int).tolist())
    out, i, last = [], 0, max(want)
    while i <= last:
        ok, f = cap.read()
        if not ok:
            break
        if i in want:
            out.append((i, f))
        i += 1
    cap.release()
    return out

rows = []
for model_name in model_folders:
    files = list_media(os.path.join(BASE_DIR, model_name), IMG_EXT + VID_EXT)
    for fpath in tqdm(files, desc=model_name):
        is_img = fpath.lower().endswith(IMG_EXT)
        frames = [(0, cv2.imread(fpath))] if is_img else read_uniform(fpath, N_FRAMES)
        for idx, fr in frames:
            if fr is None:
                continue
            crop = face_crop(fr)
            row = {"modello": model_name, "file": os.path.basename(fpath),
                   "tipo": "image" if is_img else "video", "frame": idx,
                   "frame_h": fr.shape[0], "frame_w": fr.shape[1],
                   "face_found": crop is not None,
                   "crop_h": crop.shape[0] if crop is not None else np.nan,
                   "crop_w": crop.shape[1] if crop is not None else np.nan}
            row.update(score_bgr(crop) if crop is not None else {m: np.nan for m in METRICS})
            rows.append(row)

df_frames = pd.DataFrame(rows)
df_frames.to_csv(os.path.join(SAVE_DIR, "IQA_per_frame.csv"), index=False)
print("Righe salvate:", len(df_frames))

if errors:
    print("ATTENZIONE, errori nel calcolo delle metriche:", dict(errors))
    for k, v in first_err.items():
        print("  ", k, "->", v)

In [ ]:
# Un valore per file (media dei frame), poi statistiche tra i file
per_file = (df_frames.groupby(["modello", "file", "tipo"])
            .agg(**{m: (m, "mean") for m in METRICS},
                 face_rate=("face_found", "mean"),
                 crop_h=("crop_h", "median"), crop_w=("crop_w", "median"))
            .reset_index())
per_file.to_csv(os.path.join(SAVE_DIR, "IQA_per_file.csv"), index=False)

summary = []
for mod, g in per_file.groupby("modello"):
    r = {"modello": mod, "n_file": len(g),
         "volto_trovato_%": round(100 * df_frames[df_frames.modello == mod].face_found.mean(), 1),
         "crop_mediano_px": f"{g.crop_h.median():.0f}x{g.crop_w.median():.0f}"}
    for m in METRICS:
        arrow = "↓" if m in LOWER_IS_BETTER else "↑"
        r[f"{m} {arrow}"] = f"{g[m].mean():.3f} ± {g[m].std():.3f}"
    summary.append(r)
df_summary = pd.DataFrame(summary).set_index("modello")
df_summary.to_csv(os.path.join(SAVE_DIR, "IQA_riepilogo.csv"))
print("=== RIEPILOGO IQA (media ± dev.std tra i file) ===")
display(df_summary.T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, m in zip(axes.ravel(), METRICS):
    data = [per_file.loc[per_file.modello == mod, m].dropna() for mod in model_folders]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(model_folders) + 1))
    ax.set_xticklabels(model_folders, rotation=15)
    ax.set_title(f"{m} ({'↓' if m in LOWER_IS_BETTER else '↑'} meglio)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "IQA_boxplot.png"), dpi=200)
plt.show()

# Test di Mann-Whitney tra i modelli video
vid_models = list(per_file[per_file.tipo == "video"].modello.unique())
for i in range(len(vid_models)):
    for j in range(i + 1, len(vid_models)):
        a, b = vid_models[i], vid_models[j]
        print(f"\nMann-Whitney {a} vs {b}")
        for m in METRICS:
            x = per_file.loc[per_file.modello == a, m].dropna()
            y = per_file.loc[per_file.modello == b, m].dropna()
            if len(x) > 3 and len(y) > 3:
                print(f"  {m:14s} p = {mannwhitneyu(x, y).pvalue:.4g}")

In [ ]:
import lpips

K_UNIFORM = 20     # frame sparsi su tutto il video (identity drift)
N_WINDOWS = 5      # finestre di frame CONSECUTIVI (jitter, flicker, CSIM adiacente)
WIN = 8            # frame per finestra
SOURCE_DIR = None  # opzionale: cartella con le immagini sorgente (stesso nome del video, senza estensione)

lpips_fn = lpips.LPIPS(net="alex").to(device).eval()

def norm_landmarks(lm):
    lm = np.asarray(lm, dtype=np.float64)
    lm = lm - lm.mean(0)
    return lm / (np.sqrt((lm ** 2).sum(1).mean()) + 1e-8)

def analyze(frame):
    faces = face_app.get(frame)
    if not faces:
        return None
    f = main_face(faces)
    lm = f.landmark_3d_68
    return {"emb": f.normed_embedding,
            "lm": norm_landmarks(lm) if lm is not None else None}

def lpips_tensor(bgr, max_side=512):
    h, w = bgr.shape[:2]
    s = max_side / max(h, w)
    if s < 1:
        bgr = cv2.resize(bgr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return (torch.from_numpy(rgb).permute(2, 0, 1).float() / 127.5 - 1.0).unsqueeze(0).to(device)

def sample_video(path):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 3:
        cap.release()
        return None
    uni = sorted(set(np.linspace(0, total - 1, min(K_UNIFORM, total)).astype(int).tolist()))
    starts = sorted(set(np.linspace(0, max(total - WIN, 0), N_WINDOWS).astype(int).tolist()))
    wins = [list(range(s, min(s + WIN, total))) for s in starts]
    need = set(uni)
    for w in wins:
        need.update(w)
    frames, i, last = {}, 0, max(need)
    while i <= last:
        ok, f = cap.read()
        if not ok:
            break
        if i in need:
            frames[i] = f
        i += 1
    cap.release()
    return uni, wins, frames

def nanmean(x):
    return float(np.mean(x)) if len(x) else np.nan

def source_embedding(stem):
    if SOURCE_DIR is None:
        return None
    for ext in IMG_EXT:
        p = os.path.join(SOURCE_DIR, stem + ext)
        if os.path.exists(p):
            a = analyze(cv2.imread(p))
            return None if a is None else a["emb"]
    return None

@torch.no_grad()
def evaluate_video(path):
    s = sample_video(path)
    if s is None:
        return None
    uni, wins, frames = s
    info = {i: analyze(frames[i]) for i in frames}
    emb = lambda i: None if info.get(i) is None else info[i]["emb"]
    lmk = lambda i: None if info.get(i) is None else info[i]["lm"]

    # 1) Identity drift sui frame sparsi (riferimento: primo frame con volto)
    e_list = [emb(i) for i in uni if emb(i) is not None]
    sims = [float(e_list[0] @ e) for e in e_list[1:]]
    out = {"CSIM_drift": nanmean(sims), "CSIM_drift_min": float(min(sims)) if sims else np.nan}

    stem = os.path.splitext(os.path.basename(path))[0]
    src = source_embedding(stem)
    if src is not None:
        out["CSIM_source"] = nanmean([float(src @ e) for e in e_list])

    # 2) Metriche temporali dentro finestre di frame consecutivi
    csim_adj, flick, jit, mot, cache = [], [], [], [], {}
    def lt(i):
        if i not in cache:
            cache[i] = lpips_tensor(frames[i])
        return cache[i]

    for w in wins:
        w = [i for i in w if i in frames]
        for a, b in zip(w[:-1], w[1:]):
            ea, eb = emb(a), emb(b)
            if ea is not None and eb is not None:
                csim_adj.append(float(ea @ eb))
            flick.append(float(lpips_fn(lt(a), lt(b)).item()))
            la, lb = lmk(a), lmk(b)
            if la is not None and lb is not None:
                mot.append(float(np.linalg.norm(lb - la, axis=1).mean()))
        for a, b, c in zip(w[:-2], w[1:-1], w[2:]):
            la, lb, lc = lmk(a), lmk(b), lmk(c)
            if la is not None and lb is not None and lc is not None:
                jit.append(float(np.linalg.norm(lc - 2 * lb + la, axis=1).mean()))

    out.update({"CSIM_adj": nanmean(csim_adj),
                "jitter": nanmean(jit),
                "motion": nanmean(mot),
                "flicker_lpips": nanmean(flick),
                "face_rate": float(np.mean([info[i] is not None for i in info]))})
    out["jitter_ratio"] = out["jitter"] / (out["motion"] + 1e-8) if mot and jit else np.nan
    return out

video_rows, vid_err = [], Counter()
for model_name in model_folders:
    vids = list_media(os.path.join(BASE_DIR, model_name), VID_EXT)
    if not vids:
        continue   # Reface (solo immagini) escluso
    for vpath in tqdm(vids, desc=f"Video {model_name}"):
        try:
            r = evaluate_video(vpath)
        except Exception as e:
            vid_err[repr(e)[:150]] += 1
            continue
        if r:
            r.update({"modello": model_name, "file": os.path.basename(vpath)})
            video_rows.append(r)

df_vid = pd.DataFrame(video_rows)
df_vid.to_csv(os.path.join(SAVE_DIR, "VIDEO_per_file.csv"), index=False)
print("Video valutati:", len(df_vid))
if vid_err:
    print("ATTENZIONE, video falliti:", dict(vid_err))

In [ ]:
VIDEO_METRICS = {"CSIM_drift": "↑", "CSIM_drift_min": "↑", "CSIM_adj": "↑",
                 "jitter": "↓", "jitter_ratio": "↓", "flicker_lpips": "↓",
                 "motion": "·", "face_rate": "↑"}
if "CSIM_source" in df_vid.columns:
    VIDEO_METRICS["CSIM_source"] = "↑"

summ = []
for mod, g in df_vid.groupby("modello"):
    r = {"modello": mod, "n_video": len(g)}
    for m, arrow in VIDEO_METRICS.items():
        r[f"{m} {arrow}"] = f"{g[m].mean():.4f} ± {g[m].std():.4f}"
    summ.append(r)
df_vsum = pd.DataFrame(summ).set_index("modello")
df_vsum.to_csv(os.path.join(SAVE_DIR, "VIDEO_riepilogo.csv"))
print("=== RIEPILOGO VIDEO (media ± dev.std tra i video; '·' = indicatore del movimento, non di qualità) ===")
display(df_vsum.T)

vmods = list(df_vid.modello.unique())
cols = list(VIDEO_METRICS)
nrows = (len(cols) + 3) // 4
fig, axes = plt.subplots(nrows, 4, figsize=(16, 4 * nrows))
for ax, m in zip(axes.ravel(), cols):
    data = [df_vid.loc[df_vid.modello == mod, m].dropna() for mod in vmods]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(vmods) + 1))
    ax.set_xticklabels(vmods, rotation=15)
    ax.set_title(f"{m} ({VIDEO_METRICS[m]})")
for ax in axes.ravel()[len(cols):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "VIDEO_boxplot.png"), dpi=200)
plt.show()

for i in range(len(vmods)):
    for j in range(i + 1, len(vmods)):
        a, b = vmods[i], vmods[j]
        print(f"\nMann-Whitney {a} vs {b}")
        for m in cols:
            x = df_vid.loc[df_vid.modello == a, m].dropna()
            y = df_vid.loc[df_vid.modello == b, m].dropna()
            if len(x) > 3 and len(y) > 3:
                print(f"  {m:16s} p = {mannwhitneyu(x, y).pvalue:.4g}")